# BAFERE: scam SMS detection in English, Luganda and Lusoga

Evaluation notebook for the AfriLang AI 2026 submission. It runs on a **free Colab T4 GPU**.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Then run the cells in order.

Each model step saves a checkpoint file (`results/*.csv`), so if Colab disconnects you can re-run
from the top and finished steps are skipped. For extra safety, mount Google Drive in cell 1.

| Step | What it does | Time on T4 |
|---|---|---|
| 1 | Install and load your two completed workbooks | 3 min |
| 2 | Annotator agreement | seconds |
| 3 | Classical baseline (TF-IDF trained on English spam) | 1 min |
| 4 | Sunflower-14B: direct classification + translation to English | ~45 min |
| 5 | Qwen3-14B (same base, no Ugandan adaptation) | ~25 min |
| 6 | Results tables, significance tests and figures | seconds |

In [ ]:
# ---- 1a. Install --------------------------------------------------------------
!pip -q install -U "transformers>=4.51" accelerate bitsandbytes sacrebleu datasets openpyxl

# Optional but recommended: save results to Google Drive so a disconnect loses nothing.
USE_DRIVE = True
import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/BAFERE"
else:
    WORK = "/content/BAFERE"
os.makedirs(f"{WORK}/results", exist_ok=True)
os.makedirs(f"{WORK}/figures", exist_ok=True)
print("Working folder:", WORK)

In [ ]:
# ---- 1b. Upload the two completed workbooks ------------------------------------
# Upload BAFERE_master.xlsx (with your Lusoga) and BAFERE_annotator_pack.xlsx (with the Luganda).
import shutil
MASTER = f"{WORK}/BAFERE_master.xlsx"
PACK = f"{WORK}/BAFERE_annotator_pack.xlsx"
if not (os.path.exists(MASTER) and os.path.exists(PACK)):
    from google.colab import files
    up = files.upload()
    for name in up:
        if "master" in name.lower():
            shutil.move(name, MASTER)
        elif "annotator" in name.lower():
            shutil.move(name, PACK)
print(os.path.exists(MASTER), os.path.exists(PACK))

In [ ]:
# ---- 1c. Build the dataset -----------------------------------------------------
import pandas as pd, numpy as np

def load_bafere(master_path, pack_path):
    src = pd.read_excel(master_path, sheet_name="Source_EN", skiprows=[1, 2])  # drop example + spacer rows
    src = src[src["ID"].astype(str).str.match(r"^[SL]\d-\d\d$")].copy()
    # Use Daniel's edited English where he marked EDIT
    edited = src["Realistic?"].astype(str).str.upper().eq("EDIT") & src["Suggested edit"].notna()
    src["text_en"] = np.where(edited, src["Suggested edit"], src["English message"])
    xog = pd.read_excel(master_path, sheet_name="Lusoga", skiprows=[1, 2])
    xog = xog[xog["ID"].astype(str).str.match(r"^[SL]\d-\d\d$")]
    key = pd.read_excel(master_path, sheet_name="Key")
    key.columns = ["blind", "ID"]
    lab = pd.read_excel(pack_path, sheet_name="1_Label", skiprows=[1, 2])
    lab = lab[lab["Message no."].astype(str).str.match(r"^B\d{3}$")]
    lug = pd.read_excel(pack_path, sheet_name="2_Luganda", skiprows=[1, 2])
    lug = lug[lug["Message no."].astype(str).str.match(r"^B\d{3}$")]
    lab = lab.merge(key, left_on="Message no.", right_on="blind")
    lug = lug.merge(key, left_on="Message no.", right_on="blind")
    df = pd.DataFrame({
        "id": src["ID"].values,
        "category": src["Category"].values,
        "label": src["Label"].values,
        "text_eng": src["text_en"].values,
        "edited_by_author": edited.values,
    })
    df = df.merge(xog[["ID", "Lusoga translation", "Unsure?"]].rename(
        columns={"ID": "id", "Lusoga translation": "text_xog", "Unsure?": "unsure_xog"}), on="id", how="left")
    df = df.merge(lug[["ID", "Luganda translation", "Unsure?"]].rename(
        columns={"ID": "id", "Luganda translation": "text_lug", "Unsure?": "unsure_lug"}), on="id", how="left")
    df = df.merge(lab[["ID", "SCAM or LEGIT", "Confidence (1-3)"]].rename(
        columns={"ID": "id", "SCAM or LEGIT": "annot_label", "Confidence (1-3)": "annot_conf"}), on="id", how="left")
    df["y"] = (df["label"] == "SCAM").astype(int)
    return df

df = load_bafere(MASTER, PACK)
missing = {c: int(df[c].isna().sum()) for c in ["text_eng", "text_lug", "text_xog", "annot_label"]}
print(len(df), "messages | missing:", missing)
assert len(df) == 120, "expected 120 messages"
if any(missing.values()):
    print("WARNING: some cells are empty. Those rows are dropped for the affected language only.")
df.to_csv(f"{WORK}/results/bafere_dataset.csv", index=False)
df.head(3)

In [ ]:
# ---- 2. Annotator agreement ------------------------------------------------------
from sklearn.metrics import cohen_kappa_score, confusion_matrix

def agreement(df):
    a = df.dropna(subset=["annot_label"])
    k = cohen_kappa_score(a["label"], a["annot_label"])
    acc = (a["label"] == a["annot_label"]).mean()
    dis = a[a["label"] != a["annot_label"]][["id", "label", "annot_label", "annot_conf", "text_eng"]]
    return k, acc, dis

kappa, raw_agree, disagreements = agreement(df)
print(f"Cohen's kappa (gold vs independent annotator): {kappa:.3f}   raw agreement: {raw_agree:.1%}")
print(f"Translator marked 'unsure': Luganda {int((df['unsure_lug']=='Y').sum())}, Lusoga {int((df['unsure_xog']=='Y').sum())}")
disagreements.to_csv(f"{WORK}/results/disagreements.csv", index=False)
disagreements

In [ ]:
# ---- 3. Classical baseline: TF-IDF + logistic regression trained on English SMS spam ----
# Represents an off-the-shelf spam filter: trained on the UCI SMS Spam Collection (UK, English).
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline, make_union

LANGS = {"eng": "English", "lug": "Luganda", "xog": "Lusoga"}

def train_tfidf():
    uci = load_dataset("ucirvine/sms_spam", split="train")
    X, y = uci["sms"], uci["label"]
    clf = make_pipeline(
        make_union(TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True),
                   TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        LogisticRegression(max_iter=2000, class_weight="balanced"))
    clf.fit(X, y)
    return clf

tfidf = train_tfidf()
rows = []
for lg in LANGS:
    sub = df.dropna(subset=[f"text_{lg}"])
    p = tfidf.predict_proba(sub[f"text_{lg}"].astype(str).tolist())[:, 1]
    rows += [{"id": i, "system": "TF-IDF (UCI English spam)", "condition": lg, "p_scam": v} for i, v in zip(sub["id"], p)]
pd.DataFrame(rows).to_csv(f"{WORK}/results/scores_tfidf.csv", index=False)
print("saved TF-IDF scores")

In [ ]:
# If a model download asks for a token, run: from huggingface_hub import login; login()
# ---- 4/5 shared: model loading, scoring and translation ---------------------------
import torch, gc, re, math
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

SYSTEM = "You are a fraud analyst protecting mobile money users in Uganda."
TASK = (
    "Classify the SMS message below as SCAM or LEGIT.\n"
    "SCAM: the message tries, under a false or unverifiable pretext, to get the reader to send money, airtime or a fee "
    "to a personal number or unofficial channel; reveal a PIN, one-time code or password; open an unofficial link or call "
    "an unofficial number to claim, fix or unlock something; or take an action that helps someone take over their account.\n"
    "LEGIT: anything else, including genuine transaction receipts, promotions, security warnings and ordinary requests "
    "between people who know each other.\n"
    "The message may be written in English, Luganda or Lusoga.\n\n"
    'Message: """{text}"""\n\n'
    "Answer with one word: SCAM or LEGIT."
)
CANDS = ["SCAM", "LEGIT"]

def load_model(name):
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(
        name, device_map="auto",
        quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                               bnb_4bit_compute_dtype=torch.float16))
    model.eval()
    return tok, model

def chat_prompt(tok, system, user):
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    try:
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def p_scam(tok, model, text):
    """Normalised probability of the SCAM answer, from the log-likelihood of each full answer string."""
    prompt_ids = tok(chat_prompt(tok, SYSTEM, TASK.format(text=text)), add_special_tokens=False)["input_ids"]
    lps = []
    for cand in CANDS:
        cand_ids = tok(cand, add_special_tokens=False)["input_ids"]
        ids = torch.tensor([prompt_ids + cand_ids], device=model.device)
        logits = model(ids).logits[0, len(prompt_ids) - 1:-1].float()
        logp = torch.log_softmax(logits, -1)
        lps.append(sum(logp[j, t].item() for j, t in enumerate(cand_ids)))
    return 1 / (1 + math.exp(lps[1] - lps[0]))

TR_SYSTEM = "You are a linguist and translator specializing in Ugandan languages, made by Sunbird AI."

@torch.no_grad()
def translate_to_english(tok, model, text, lang_name):
    user = f"Translate the following {lang_name} text to English. Output only the translation.\n\n{text}"
    ids = tok(chat_prompt(tok, TR_SYSTEM, user), return_tensors="pt", add_special_tokens=False).to(model.device)
    out = model.generate(**ids, max_new_tokens=160, do_sample=False, pad_token_id=tok.eos_token_id)
    s = tok.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True)
    s = re.sub(r"<think>.*?</think>", "", s, flags=re.S).strip()
    return s.split("\n\n")[0].strip()

def score_conditions(tok, model, system_name, conditions, ckpt):
    """conditions: dict condition_name -> DataFrame with columns id,text. Resumes from checkpoint."""
    done = pd.read_csv(ckpt) if os.path.exists(ckpt) else pd.DataFrame(columns=["id", "system", "condition", "p_scam"])
    rows = done.to_dict("records")
    seen = {(r["condition"], r["id"]) for r in rows}
    for cond, sub in conditions.items():
        for i, (mid, text) in enumerate(zip(sub["id"], sub["text"])):
            if (cond, mid) in seen:
                continue
            rows.append({"id": mid, "system": system_name, "condition": cond, "p_scam": p_scam(tok, model, str(text))})
            if len(rows) % 20 == 0:
                pd.DataFrame(rows).to_csv(ckpt, index=False)
        pd.DataFrame(rows).to_csv(ckpt, index=False)
        print(f"  {system_name} / {cond}: done")
    return pd.DataFrame(rows)

def direct_conditions(df):
    return {lg: df.dropna(subset=[f"text_{lg}"])[["id", f"text_{lg}"]].rename(columns={f"text_{lg}": "text"})
            for lg in LANGS}

In [ ]:
# ---- 4. Sunflower-14B (Ugandan-language adaptation of Qwen3-14B) ---------------------
SUNFLOWER = "Sunbird/Sunflower-14B"
tok, model = load_model(SUNFLOWER)

# 4a. Direct classification in each language
score_conditions(tok, model, "Sunflower-14B", direct_conditions(df), f"{WORK}/results/scores_sunflower.csv")

# 4b. Translate Luganda and Lusoga to English (translate-test), with resume
TR_CKPT = f"{WORK}/results/translations_sunflower.csv"
tr = pd.read_csv(TR_CKPT) if os.path.exists(TR_CKPT) else pd.DataFrame(columns=["id", "src_lang", "mt_eng"])
have = set(zip(tr["src_lang"], tr["id"]))
new = []
for lg in ["lug", "xog"]:
    sub = df.dropna(subset=[f"text_{lg}"])
    for mid, text in zip(sub["id"], sub[f"text_{lg}"]):
        if (lg, mid) in have:
            continue
        new.append({"id": mid, "src_lang": lg, "mt_eng": translate_to_english(tok, model, str(text), LANGS[lg])})
        if len(new) % 10 == 0:
            pd.concat([tr, pd.DataFrame(new)]).to_csv(TR_CKPT, index=False)
    print(f"  translated {LANGS[lg]}")
tr = pd.concat([tr, pd.DataFrame(new)], ignore_index=True)
tr.to_csv(TR_CKPT, index=False)

def tt_conditions(tr):
    return {f"{lg}->eng (MT)": tr[tr["src_lang"] == lg][["id", "mt_eng"]].rename(columns={"mt_eng": "text"})
            for lg in ["lug", "xog"]}

# 4c. Classify the machine translations
score_conditions(tok, model, "Sunflower-14B", tt_conditions(tr), f"{WORK}/results/scores_sunflower.csv")
del model, tok; gc.collect(); torch.cuda.empty_cache()
tr.head()

In [ ]:
# ---- 5. Qwen3-14B: identical base model without Ugandan-language adaptation -------------
QWEN = "Qwen/Qwen3-14B"
tr = pd.read_csv(TR_CKPT)
tok, model = load_model(QWEN)
conds = {**direct_conditions(df), **tt_conditions(tr)}
score_conditions(tok, model, "Qwen3-14B", conds, f"{WORK}/results/scores_qwen3.csv")
del model, tok; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ---- 6a. Metrics with bootstrap confidence intervals ---------------------------------
from sklearn.metrics import f1_score, roc_auc_score  # used to cross-check the fast versions
from scipy.stats import binomtest
import sacrebleu, json

scores = pd.concat([pd.read_csv(f"{WORK}/results/{f}") for f in
                    ["scores_tfidf.csv", "scores_sunflower.csv", "scores_qwen3.csv"]
                    if os.path.exists(f"{WORK}/results/{f}")], ignore_index=True)
scores = scores.merge(df[["id", "y", "category"]], on="id")
scores["pred"] = (scores["p_scam"] >= 0.5).astype(int)
scores["correct"] = (scores["pred"] == scores["y"]).astype(int)

def _macro_f1(y, pred):
    f = []
    for c in (0, 1):
        tp = np.sum((pred == c) & (y == c)); fp = np.sum((pred == c) & (y != c)); fn = np.sum((pred != c) & (y == c))
        f.append(0.0 if tp == 0 else 2 * tp / (2 * tp + fp + fn))
    return float(np.mean(f))

def _auroc(y, p):
    """Mann-Whitney form of AUROC, ties counted as half."""
    pos, neg = p[y == 1], p[y == 0]
    if len(pos) == 0 or len(neg) == 0:
        return float("nan")
    diff = pos[:, None] - neg[None, :]
    return float(((diff > 0).sum() + 0.5 * (diff == 0).sum()) / diff.size)

def metrics(y, p, pred):
    return {"n": len(y),
            "accuracy": float((pred == y).mean()),
            "macro_f1": _macro_f1(y, pred),
            "scam_recall": float(pred[y == 1].mean()),
            "false_alarm_rate": float(pred[y == 0].mean()),
            "auroc": _auroc(y, p)}

def bootstrap_ci(y, p, pred, B=2000, seed=0):
    """Stratified bootstrap (keeps the 60/60 balance). Returns 95% intervals for macro-F1 and AUROC."""
    rng = np.random.default_rng(seed)
    idx1, idx0 = np.where(y == 1)[0], np.where(y == 0)[0]
    f1s, aucs = [], []
    for _ in range(B):
        s = np.concatenate([rng.choice(idx1, len(idx1)), rng.choice(idx0, len(idx0))])
        f1s.append(_macro_f1(y[s], pred[s])); aucs.append(_auroc(y[s], p[s]))
    return np.percentile(f1s, [2.5, 97.5]), np.percentile(aucs, [2.5, 97.5])

ORDER = ["eng", "lug", "xog", "lug->eng (MT)", "xog->eng (MT)"]
table = []
for (sysname, cond), g in scores.groupby(["system", "condition"]):
    y, p, pred = g["y"].values, g["p_scam"].values, g["pred"].values
    m = metrics(y, p, pred)
    (lo, hi), (alo, ahi) = bootstrap_ci(y, p, pred)
    table.append({"system": sysname, "condition": cond, **m,
                  "f1_lo": lo, "f1_hi": hi, "auroc_lo": alo, "auroc_hi": ahi})
table = pd.DataFrame(table)
table["condition"] = pd.Categorical(table["condition"], ORDER, ordered=True)
table = table.sort_values(["system", "condition"])
table.to_csv(f"{WORK}/results/main_table.csv", index=False)
pd.options.display.float_format = "{:.3f}".format
table

In [ ]:
# ---- 6b. Language gap significance (exact McNemar on paired messages) -----------------
def mcnemar(a, b):
    """a, b: 0/1 correctness on the same messages. Exact two-sided test on discordant pairs."""
    n01 = int(((a == 1) & (b == 0)).sum()); n10 = int(((a == 0) & (b == 1)).sum())
    pval = binomtest(n01, n01 + n10, 0.5).pvalue if n01 + n10 else 1.0
    return n01, n10, pval

tests = []
for sysname, g in scores.groupby("system"):
    piv = g.pivot(index="id", columns="condition", values="correct")
    for other in ["lug", "xog", "lug->eng (MT)", "xog->eng (MT)"]:
        if "eng" in piv and other in piv:
            both = piv[["eng", other]].dropna()
            n01, n10, pv = mcnemar(both["eng"].values, both[other].values)
            tests.append({"system": sysname, "comparison": f"eng vs {other}", "eng_only_correct": n01,
                          "other_only_correct": n10, "p_value": pv})
    for lg in ["lug", "xog"]:  # does translate-test beat direct classification?
        mt = f"{lg}->eng (MT)"
        if lg in piv and mt in piv:
            both = piv[[lg, mt]].dropna()
            n01, n10, pv = mcnemar(both[lg].values, both[mt].values)
            tests.append({"system": sysname, "comparison": f"{lg} direct vs {mt}", "eng_only_correct": n01,
                          "other_only_correct": n10, "p_value": pv})
tests = pd.DataFrame(tests)
tests.to_csv(f"{WORK}/results/significance.csv", index=False)
tests

In [ ]:
# ---- 6c. Where do errors fall? Accuracy by fraud type and legit category --------------
cat = scores[scores["condition"].isin(["eng", "lug", "xog"])].groupby(
    ["system", "condition", "category"])["correct"].mean().unstack("category")
cat.to_csv(f"{WORK}/results/by_category.csv")
cat

In [ ]:
# ---- 6d. Translation quality: Sunflower MT back to English vs the original English source ----
tr = pd.read_csv(TR_CKPT).merge(df[["id", "text_eng"]], on="id")
chrf = {lg: sacrebleu.corpus_chrf(tr[tr.src_lang == lg]["mt_eng"].astype(str).tolist(),
                                  [tr[tr.src_lang == lg]["text_eng"].astype(str).tolist()]).score
        for lg in ["lug", "xog"]}
print("chrF of Sunflower translation vs original English:", {k: round(v, 1) for k, v in chrf.items()})
json.dump({"kappa": kappa, "raw_agreement": raw_agree, "chrf": chrf},
          open(f"{WORK}/results/summary.json", "w"), indent=2)

In [ ]:
# ---- 6e. Figure: macro-F1 by system and language, with 95% bootstrap intervals ----------
import matplotlib.pyplot as plt
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 9, "axes.edgecolor": "#8a8984",
                     "axes.labelcolor": "#0b0b0b", "xtick.color": "#52514e", "ytick.color": "#52514e"})
COL = {"eng": "#2a78d6", "lug": "#eb6834", "xog": "#1baf7a"}  # validated categorical slots 1-3
NAME = {"eng": "English", "lug": "Luganda", "xog": "Lusoga"}

def fig_f1(table, path):
    systems = [s for s in ["TF-IDF (UCI English spam)", "Qwen3-14B", "Sunflower-14B"] if s in set(table.system)]
    fig, ax = plt.subplots(figsize=(6.2, 2.9))
    w = 0.26
    for j, lg in enumerate(["eng", "lug", "xog"]):
        t = table[table.condition == lg].set_index("system").reindex(systems)
        x = np.arange(len(systems)) + (j - 1) * w
        ax.bar(x, t["macro_f1"], width=w - 0.03, color=COL[lg], label=NAME[lg], zorder=2)
        ax.errorbar(x, t["macro_f1"], yerr=[t["macro_f1"] - t["f1_lo"], t["f1_hi"] - t["macro_f1"]],
                    fmt="none", ecolor="#3d3d3a", elinewidth=1, capsize=2, zorder=3)
    ax.set_xticks(np.arange(len(systems)), systems)
    ax.set_ylim(0, 1); ax.set_ylabel("Macro-F1")
    ax.axhline(0.5, color="#bdbcb5", lw=0.8, ls="--", zorder=1)
    ax.text(len(systems) - 0.5, 0.51, "chance", ha="right", va="bottom", fontsize=7, color="#52514e")
    ax.grid(axis="y", color="#e8e7e2", lw=0.6, zorder=0)
    for s in ["top", "right"]:
        ax.spines[s].set_visible(False)
    ax.legend(frameon=False, ncol=3, loc="upper left", bbox_to_anchor=(0, 1.13))
    fig.tight_layout(); fig.savefig(path, bbox_inches="tight"); plt.show()

fig_f1(table, f"{WORK}/figures/f1_by_language.pdf")

In [ ]:
# ---- 6f. Dataset statistics: length and tokenizer fertility per language ----------------
# Fertility = subword tokens per whitespace word under the Qwen3 tokenizer (shared by Sunflower).
from transformers import AutoTokenizer
qtok = AutoTokenizer.from_pretrained("Qwen/Qwen3-14B")
stats = {}
for lg in LANGS:
    t = df[f"text_{lg}"].dropna().astype(str)
    words = t.str.split().str.len()
    toks = t.apply(lambda x: len(qtok(x, add_special_tokens=False)["input_ids"]))
    stats[lg] = {"n": int(len(t)), "words_mean": float(words.mean()), "tokens_mean": float(toks.mean()),
                 "fertility": float(toks.sum() / words.sum())}
json.dump(stats, open(f"{WORK}/results/dataset_stats.json", "w"), indent=2)
pd.DataFrame(stats).T

In [ ]:
# ---- 6g. Export every number the paper uses as LaTeX macros -----------------------------
SYS = {"TF-IDF (UCI English spam)": "Tfidf", "Qwen3-14B": "Qwen", "Sunflower-14B": "Sun"}
CND = {"eng": "Eng", "lug": "Lug", "xog": "Xog", "lug->eng (MT)": "LugMT", "xog->eng (MT)": "XogMT"}
f2 = lambda v: "--" if pd.isna(v) else f"{v:.2f}"
pct = lambda v: "--" if pd.isna(v) else f"{100 * v:.0f}\\%"
def pfmt(p):
    return "--" if pd.isna(p) else ("<0.001" if p < 0.001 else f"{p:.3f}")
lines = ["% Auto-generated by BAFERE_evaluation.ipynb. Do not edit by hand."]
def mac(name, val):
    lines.append(f"\\renewcommand{{\\{name}}}{{{val}}}")
for _, r in table.iterrows():
    if r["system"] not in SYS: continue
    k = SYS[r["system"]] + CND[str(r["condition"])]
    mac(k + "Fone", f2(r["macro_f1"])); mac(k + "Flo", f2(r["f1_lo"])); mac(k + "Fhi", f2(r["f1_hi"]))
    mac(k + "Auc", f2(r["auroc"])); mac(k + "Rec", pct(r["scam_recall"])); mac(k + "Far", pct(r["false_alarm_rate"]))
    mac(k + "N", str(int(r["n"])))
for _, r in tests.iterrows():
    if r["system"] not in SYS: continue
    c = r["comparison"]
    if c.startswith("eng vs "):
        k = SYS[r["system"]] + "EngVs" + CND[c[len("eng vs "):]]
    else:
        lg = c.split()[0]; k = SYS[r["system"]] + CND[lg] + "DirVsMT"
    mac(k + "P", pfmt(r["p_value"]))
mac("resKappa", f2(kappa)); mac("resAgree", pct(raw_agree))
mac("resChrfLug", f"{chrf['lug']:.1f}"); mac("resChrfXog", f"{chrf['xog']:.1f}")
for lg, st in stats.items():
    L = CND[lg]
    mac(f"stat{L}Words", f"{st['words_mean']:.1f}"); mac(f"stat{L}Tokens", f"{st['tokens_mean']:.1f}")
    mac(f"stat{L}Fert", f"{st['fertility']:.2f}"); mac(f"stat{L}N", str(st["n"]))
mac("resUnsureLug", str(int((df["unsure_lug"] == "Y").sum()))); mac("resUnsureXog", str(int((df["unsure_xog"] == "Y").sum())))
mac("resEdited", str(int(df["edited_by_author"].sum())))
open(f"{WORK}/results/results_macros.tex", "w").write("\n".join(lines) + "\n")

# Accuracy by category for the two LLMs, as a LaTeX table
CATS = ["S1", "S2", "S3", "S4", "S5", "S6", "L1", "L2", "L3", "L4", "L5", "L6"]
rows_tex = []
for sysname in ["Qwen3-14B", "Sunflower-14B"]:
    for lg in ["eng", "lug", "xog"]:
        if (sysname, lg) not in cat.index: continue
        vals = cat.loc[(sysname, lg)].reindex(CATS)
        rows_tex.append(f"{sysname if lg == 'eng' else ''} & {NAME[lg]} & " +
                        " & ".join("--" if pd.isna(v) else f"{100 * v:.0f}" for v in vals) + r" \\")
    rows_tex.append(r"\midrule")
tex = (r"\begin{tabular}{ll" + "r" * 12 + "}\n\\toprule\n & & \\multicolumn{6}{c}{Scam types} & \\multicolumn{6}{c}{Legitimate categories} \\\\\n"
       r"\cmidrule(lr){3-8}\cmidrule(lr){9-14}" + "\nSystem & Lang. & " + " & ".join(CATS) + r" \\" + "\n\\midrule\n" +
       "\n".join(rows_tex[:-1]) + "\n\\bottomrule\n\\end{tabular}\n")
open(f"{WORK}/results/table_category.tex", "w").write(tex)
print("\n".join(lines[:12]), "\n...", len(lines), "macros written")

In [ ]:
# ---- 6h. Package everything for the paper ---------------------------------------------
shutil.make_archive(f"/content/bafere_results", "zip", f"{WORK}")
try:
    from google.colab import files
    files.download("/content/bafere_results.zip")
except Exception as e:
    print("Download the zip from the Files panel:", e)